In [1]:
# ===== Cell 1: 环境准备（X盘 + 路径配置）=====
# 依赖说明：
#   faiss-cpu / langchain / transformers / torch / bitsandbytes 已安装在 15-transformers-basics\.venv
#   嵌入模型直接用 transformers 加载，不需要 sentence-transformers
import os
import subprocess

# 自动创建 X: 盘，映射到项目根目录（解决中文路径下 SentencePiece / 模型加载问题）
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if not os.path.exists('X:\\'):
    subprocess.run(['subst', 'X:', project_root], check=False)
    print(f'已创建 X: 盘 -> {project_root}')

# 两个本地模型的路径
EMBED_PATH = r'X:\6.transformers\models\bge-large-zh-v1.5'   # 中文向量模型
LLM_PATH   = r'X:\6.transformers\models\chatgml3-6B'          # 对话大模型
print('环境准备完成')



已创建 X: 盘 -> S:\PythonAILearningCourse
环境准备完成


In [2]:
# ===== Cell 2: 加载本地 bge-large-zh 嵌入模型（直接基于 transformers 实现）=====
import time
import torch
from transformers import AutoTokenizer, AutoModel
from langchain_core.embeddings import Embeddings


class BGEEmbeddings(Embeddings):
    '''本地 bge-large-zh 向量模型：取 <[BOS_never_used_51bce0c785ca2f68081bfa7d91973934]> 向量并做 L2 归一化，输出 1024 维'''

    def __init__(self, model_path, device='cpu'):
        # device='cpu'：嵌入模型放 CPU，把显存留给 4-bit ChatGLM（文档少，CPU 足够快）
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModel.from_pretrained(model_path).to(device).eval()

    def _embed(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        with torch.no_grad():
            inputs = self.tokenizer(
                texts, padding=True, truncation=True,
                max_length=512, return_tensors='pt',
            ).to(self.device)
            outputs = self.model(**inputs)
            cls = outputs.last_hidden_state[:, 0]                    # bge 用 <[BOS_never_used_51bce0c785ca2f68081bfa7d91973934]> 位置向量
            cls = torch.nn.functional.normalize(cls, p=2, dim=1)     # L2 归一化 -> 余弦相似度
        return cls.cpu().tolist()

    def embed_documents(self, texts):
        # LangChain 建库时批量调用
        return self._embed(texts)

    def embed_query(self, text):
        # LangChain 检索时调用，返回单条向量
        return self._embed(text)[0]


embeddings = BGEEmbeddings(EMBED_PATH, device='cpu')

# 测试：一句话 -> 1024 维向量
t0 = time.time()
vec = embeddings.embed_query('今天天气怎么样？')
print(f'嵌入模型就绪：向量维度 {len(vec)}，首次编码耗时 {time.time()-t0:.2f}s')



S:\PythonAILearningCourse\15-transformers-basics\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6826.02it/s]


嵌入模型就绪：向量维度 1024，首次编码耗时 2.22s


In [3]:
# ===== Cell 3: 构建 FAISS 向量库 + 相似度检索 =====
import time
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# 1) 准备知识库文档（模拟"星河科技"的公司/产品/售后资料）
docs = [
    Document(page_content='星河科技成立于2018年，总部位于深圳，是一家专注于智能穿戴设备的高新技术企业，现有员工约1200人，研发人员占比超过60%。', metadata={'source': '公司简介'}),
    Document(page_content='星河智能手表X1搭载1.9英寸AMOLED屏幕，支持心率、血氧、睡眠监测和100多种运动模式，满电续航长达14天，售价999元。', metadata={'source': '产品手册-手表X1'}),
    Document(page_content='星河扫地机器人S2配备激光雷达导航和5200mAh大电池，单次充电可连续工作180分钟，支持自动集尘和APP远程控制，售价1899元。', metadata={'source': '产品手册-扫地机S2'}),
    Document(page_content='星河所有产品支持7天无理由退货、15天内质量问题免费换新，整机保修一年，核心零部件保修两年。', metadata={'source': '售后政策'}),
    Document(page_content='星河智能手表X1防水等级为5ATM，可佩戴游泳，但不建议潜水或洗热水澡；运动后建议用清水冲洗表带并擦干。', metadata={'source': '常见问题'}),
    Document(page_content='星河客服热线是400-888-6666，服务时间为每天9:00到21:00；也可以在官方App内联系在线客服提交工单，工程师24小时内响应。', metadata={'source': '联系客服'}),
]

# 2) 文档 -> 向量 -> 存入 FAISS
t0 = time.time()
vectorstore = FAISS.from_documents(docs, embeddings)
print(f'向量库构建完成：{len(docs)} 篇文档，耗时 {time.time()-t0:.1f}s')

# 3) 相似度检索测试
question = '手表充一次电能用多久？'
results = vectorstore.similarity_search_with_score(question, k=3)
print(f'\n问题：{question}')
print('检索结果（score 是 L2 距离，越小越相似；向量已归一化，排序等同余弦相似度）：')
for i, (doc, score) in enumerate(results, 1):
    print(f'\n[{i}] score={score:.4f}  来源：{doc.metadata["source"]}')
    print(f'    {doc.page_content}')



C:\Users\rehuanoo\AppData\Local\Temp\ipykernel_24232\4132575372.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


向量库构建完成：6 篇文档，耗时 0.6s

问题：手表充一次电能用多久？
检索结果（score 是 L2 距离，越小越相似；向量已归一化，排序等同余弦相似度）：

[1] score=1.0997  来源：产品手册-手表X1
    星河智能手表X1搭载1.9英寸AMOLED屏幕，支持心率、血氧、睡眠监测和100多种运动模式，满电续航长达14天，售价999元。

[2] score=1.1899  来源：常见问题
    星河智能手表X1防水等级为5ATM，可佩戴游泳，但不建议潜水或洗热水澡；运动后建议用清水冲洗表带并擦干。

[3] score=1.3085  来源：售后政策
    星河所有产品支持7天无理由退货、15天内质量问题免费换新，整机保修一年，核心零部件保修两年。


In [4]:
# ===== Cell 4: 自定义 LLM —— 本地 4-bit 量化 ChatGLM3-6B =====
import torch
from typing import Any
from pydantic import PrivateAttr
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from langchain_core.language_models.llms import LLM


class ChatGLM3(LLM):
    '''接入 LangChain 的本地 ChatGLM3-6B，4-bit 量化后显存约 3.8GB'''

    model_path: str = LLM_PATH
    max_new_tokens: int = 512
    temperature: float = 0.2      # 知识库问答用低温度，答案更稳定
    top_p: float = 0.9
    _tokenizer: Any = PrivateAttr(default=None)
    _model: Any = PrivateAttr(default=None)

    @property
    def _llm_type(self) -> str:
        return 'ChatGLM3-6B-4bit'

    def model_post_init(self, _context):
        super().model_post_init(_context)
        print(f'正在加载模型：{self.model_path} ...')
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,   # 计算用 bf16
            bnb_4bit_use_double_quant=True,          # 二级量化更省显存
            bnb_4bit_quant_type='nf4',               # NF4 量化质量更好
        )
        self._tokenizer = AutoTokenizer.from_pretrained(self.model_path, trust_remote_code=True)
        self._model = AutoModel.from_pretrained(
            self.model_path,
            trust_remote_code=True,
            quantization_config=quantization_config,
        ).eval()
        print(f'模型加载完成，显存占用：{torch.cuda.memory_allocated()/1024**2:.0f} MB')

    def _call(self, prompt: str, stop=None, run_manager=None, **kwargs) -> str:
        # build_chat_input 会自动拼接 ChatGLM3 的对话格式
        inputs = self._tokenizer.build_chat_input(prompt, history=[], role='user')
        inputs = inputs.to(self._model.device)
        with torch.no_grad():
            output = self._model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=True,
                temperature=self.temperature,
                top_p=self.top_p,
                use_cache=False,
            )
        input_len = inputs['input_ids'].shape[1]
        # 只解码新生成的部分
        return self._tokenizer.decode(output[0][input_len:], skip_special_tokens=True)


# 实例化时自动加载模型（约 10~20 秒）
llm = ChatGLM3()



正在加载模型：X:\6.transformers\models\chatgml3-6B ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 200/200 [00:13<00:00, 14.36it/s]


模型加载完成，显存占用：3833 MB


In [5]:
# ===== Cell 5: 组装 RAG 检索问答链 =====
import time
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 提示词：要求模型严格依据检索到的资料回答，禁止编造
template = '''你是星河科技的智能客服助手。请严格根据下面的参考资料回答用户问题，
不要编造资料以外的信息；如果资料中没有相关答案，请直接回答"根据现有资料无法回答该问题"。

参考资料：
{context}

用户问题：{question}'''

prompt = ChatPromptTemplate.from_template(template)

# LCEL 链：提示词模板 -> 本地大模型 -> 字符串输出
chain = prompt | llm | StrOutputParser()


def ask(question, k=3):
    '''先检索相关文档，再把文档作为上下文交给大模型回答'''
    # 1. 向量检索 top-k 文档
    results = vectorstore.similarity_search_with_score(question, k=k)
    context = '\n\n'.join(
        f'[资料{i+1}]（来源：{doc.metadata["source"]}）\n{doc.page_content}'
        for i, (doc, _) in enumerate(results)
    )
    # 2. 调大模型生成回答
    t0 = time.time()
    answer = chain.invoke({'context': context, 'question': question})
    print(f'问：{question}')
    print(f'（检索 {k} 篇资料，生成耗时 {time.time()-t0:.1f}s）')
    print(f'答：{answer}\n')


# ===== 测试提问 =====
ask('智能手表X1续航怎么样？')
ask('产品保修多久？')
ask('你们公司卖汽车吗？')   # 资料中没有，模型应回答"无法回答"



问：智能手表X1续航怎么样？
（检索 3 篇资料，生成耗时 5.2s）
答：
 智能手表X1的续航表现良好，满电续航长达14天。

问：产品保修多久？
（检索 3 篇资料，生成耗时 1.7s）
答：
 产品保修一年。

问：你们公司卖汽车吗？
（检索 3 篇资料，生成耗时 10.4s）
答：
 根据提供的参考资料，星河科技是一家专注于智能穿戴设备的高新技术企业，没有涉及汽车销售业务。因此，回答用户问题：你们公司卖汽车吗？答案是“不，星河科技专注于智能穿戴设备。”

